In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
# =========================
# Project paths
# =========================
project_dir = Path.cwd()

data_dir = project_dir / "Data"
results_dir = project_dir / "Results"


# =========================
# Config
# =========================
xlsx_path = data_dir / "MSA_Key_connectomes.xlsx"
connectome_csv_path = data_dir / "Connectome_Data.csv"

out_dir = results_dir / "MSA_Key_Connectomes"
out_dir.mkdir(parents=True, exist_ok=True)

sheet_lowest = "MSA_Lowest_For_Plot"
sheet_largest = "MSA_Largest_For_Plot"

baseline_ccc = 0.395649442591783  # Full connectome
baseline_color = "#8B0000"        # Dark red


# =========================
# Output filenames
# No file extension here
# =========================
fn_lowest_zoom = out_dir / "LowestConnections_rows2to10"
fn_lowest_full = out_dir / "LowestConnections_full"
fn_largest_full = out_dir / "LargestConnections_full"


# =========================
# Axis settings
# =========================
Y_LIM = (0.0, 0.5)
Y_TICKS = np.arange(0.0, 0.51, 0.1)

In [9]:

# =========================
# Helpers
# =========================
def to_percent_if_fraction(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    if x.dropna().size and x.dropna().max() <= 1.0 + 1e-9:
        return x * 100.0
    return x

def load_sheet(path: str, sheet: str) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet)
    df = df.iloc[:, :2].copy()
    df.columns = ["Remaining", "CCC"]
    df["Remaining"] = to_percent_if_fraction(df["Remaining"])
    df["CCC"] = pd.to_numeric(df["CCC"], errors="coerce")
    df = df.dropna(subset=["Remaining", "CCC"]).sort_values("Remaining").reset_index(drop=True)
    return df

def plot_single(
    df: pd.DataFrame,
    title: str,
    save_stem: Path,
    xlim=None,
    xticks=None
):
    fig, ax = plt.subplots(figsize=(6.8, 4.2))

    ax.plot(df["Remaining"], df["CCC"], marker="o", linewidth=1.8)

    ax.axhline(baseline_ccc, linestyle="--", linewidth=1.8, color=baseline_color)
    ax.text(
        0.98,
        baseline_ccc,
        f" Full connectome = {baseline_ccc:.3f}",
        transform=ax.get_yaxis_transform(),
        ha="right",
        va="bottom",
        fontsize=10,
        color=baseline_color,
    )

    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Proportion remaining (%)", fontsize=12)
    ax.set_ylabel("Average CCC", fontsize=12)

    ax.set_ylim(*Y_LIM)
    ax.set_yticks(Y_TICKS)

    # ---- FIX: pad x-limits to avoid cutting edge markers ----
    if xlim is not None:
        x0, x1 = xlim
        span = (x1 - x0)
        pad = 0.02 * span  # 2% padding on each side
        if pad == 0:
            pad = 0.2
        ax.set_xlim(x0 - pad, x1 + pad)

    if xticks is not None:
        ax.set_xticks(xticks)
    # ---------------------------------------------------------

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(str(save_stem) + ".pdf")
    fig.savefig(str(save_stem) + ".png", dpi=300)
    plt.close(fig)


In [10]:
# =========================
# Main
# =========================
df_lowest = load_sheet(xlsx_path, sheet_lowest)
df_largest = load_sheet(xlsx_path, sheet_largest)

# 1) Lowest connections: 0%–10%
df_lowest_zoom = df_lowest[
    df_lowest["Remaining"].between(0, 10)
].copy()

zoom_xticks = [0, 2, 5, 10]

plot_single(
    df_lowest_zoom,
    title="Removing weakest connections (rows 2–10)",
    save_stem=fn_lowest_zoom,
    xlim=(0, 10),
    xticks=zoom_xticks,
)

# 2) Lowest connections: full range
lowest_full_xticks = [0, 20, 40, 60, 80, 100]

plot_single(
    df_lowest,
    title="Removing weakest connections (full)",
    save_stem=fn_lowest_full,
    xlim=(0, 100),
    xticks=lowest_full_xticks,
)

# 3) Largest connections: 80%–100%
largest_xticks = [80, 85, 90, 95, 100]

plot_single(
    df_largest,
    title="Removing strongest connections (80–100%)",
    save_stem=fn_largest_full,
    xlim=(80, 100),
    xticks=largest_xticks,
)

print("Saved to:", out_dir)
print(" -", str(fn_lowest_zoom) + ".pdf/.png")
print(" -", str(fn_lowest_full) + ".pdf/.png")
print(" -", str(fn_largest_full) + ".pdf/.png")

Saved to: C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_Key_Connectomes
 - C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_Key_Connectomes\LowestConnections_rows2to10.pdf/.png
 - C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_Key_Connectomes\LowestConnections_full.pdf/.png
 - C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_Key_Connectomes\LargestConnections_full.pdf/.png


In [27]:
# visualization of key connectomes matrix


from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcParams["pdf.compression"] = 0

def white_zero_cmap(base_cmap="Oranges", n=256):
    """
    Colormap with pure white at the lowest value (index 0),
    so log10(weight+1)=0 appears as white background.
    """
    base = cm.get_cmap(base_cmap, n)
    colors = base(np.linspace(0, 1, n))
    colors[0] = [1, 1, 1, 1]  # pure white
    return mcolors.ListedColormap(colors)


In [20]:
# =========================
# Paths
# =========================
csv_path = data_dir / "Connectome_Data.csv"

out_dir = (
    results_dir
    / "MSA_Key_Connectomes"
    / "Matrix_visualization"
)
out_dir.mkdir(parents=True, exist_ok=True)

# =========================
# Load matrix
# =========================
W = np.loadtxt(csv_path, delimiter=",")

if W.shape != (410, 410):
    raise ValueError(f"Expected (410,410), got {W.shape}")

# Unified colorbar scale from FULL matrix
W_log_full = np.log10(W + 1.0)
vmin, vmax = 0.0, float(W_log_full.max())



In [21]:

# =========================
# Global top-k selection (INCLUDING zeros)
# =========================
N = W.size
flat = W.ravel()


def top_k_mask(
    flat_values: np.ndarray,
    keep_frac: float
):
    k = int(np.ceil(keep_frac * flat_values.size))

    idx_sorted = np.argsort(flat_values)[::-1]  # descending
    keep_idx = idx_sorted[:k]

    mask = np.zeros(flat_values.size, dtype=bool)
    mask[keep_idx] = True

    return mask, idx_sorted


# Global top 5%
mask5_flat, idx_sorted = top_k_mask(flat, 0.05)

# Global top 1%
mask1_flat, _ = top_k_mask(flat, 0.01)

# Top 5% minus GLOBAL top 1%
# This retains globally ranked connections from 1% to 5%.
mask5_minus_flat = mask5_flat & (~mask1_flat)


# =========================
# Build matrices for plotting
# =========================
def build_weighted_and_binary(
    W: np.ndarray,
    mask_flat: np.ndarray
):
    mask = mask_flat.reshape(W.shape)

    W_kept = np.zeros_like(W, dtype=float)
    W_kept[mask] = W[mask]

    # Removed entries -> log10(1) = 0 (white)
    Wlog_kept = np.log10(W_kept + 1.0)

    B = np.zeros_like(W, dtype=int)
    B[mask] = 1

    return Wlog_kept, B


# Whole connectome (weighted + binary)
Wlog_whole = np.log10(W + 1.0)
B_whole = (W != 0).astype(int)

# Top 1% / Top 5% / Top 5% minus global Top 1%
Wlog_top1, B_top1 = build_weighted_and_binary(
    W,
    mask1_flat
)

Wlog_top5, B_top5 = build_weighted_and_binary(
    W,
    mask5_flat
)

Wlog_top5_minus, B_top5_minus = build_weighted_and_binary(
    W,
    mask5_minus_flat
)

In [22]:
# =========================
# Plot helper
# =========================
def save_matrix_fig(
    mat,
    title,
    cmap,
    vmin=None,
    vmax=None,
    with_colorbar=False,
    out_stem=None
):
    fig, ax = plt.subplots(
        figsize=(4.6, 4.6),
        dpi=200
    )

    im = ax.imshow(
        mat,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        origin="upper"
    )

    ax.set_xlabel("Brain Region No.")
    ax.set_ylabel("Brain Region No.")
    ax.set_title(title)

    ax.set_xticks([0, 100, 200, 300, 400])
    ax.set_yticks([0, 100, 200, 300, 400])

    # Ticks inward (publication style)
    ax.tick_params(
        axis="both",
        which="both",
        direction="in",
        length=4,
        width=1
    )

    if with_colorbar:
        cbar = fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04
        )
        cbar.set_label(
            r"$\log_{10}(\mathrm{weight}+1)$"
        )

    fig.tight_layout()

    fig.savefig(
        str(out_stem) + ".png",
        dpi=300
    )
    fig.savefig(
        str(out_stem) + ".pdf"
    )

    plt.close(fig)


# Custom colormap: 0 -> pure white
orange_white = white_zero_cmap("Oranges")


In [28]:
# =========================
# Save figures (8 total)
# =========================

# (H) Whole connectome
save_matrix_fig(
    Wlog_whole,
    title="Whole connectome (weighted)",
    cmap=orange_white,
    vmin=vmin,
    vmax=vmax,
    with_colorbar=True,
    out_stem=out_dir / "MSA_H_whole_weighted_log10"
)

save_matrix_fig(
    B_whole,
    title="Whole connectome (binary)",
    cmap="gray_r",
    vmin=0,
    vmax=1,
    with_colorbar=False,
    out_stem=out_dir / "MSA_H_whole_binary"
)


# (G) Global top 1%
save_matrix_fig(
    Wlog_top1,
    title="Top 1% strongest connections (weighted)",
    cmap=orange_white,
    vmin=vmin,
    vmax=vmax,
    with_colorbar=True,
    out_stem=out_dir / "MSA_G_top1_weighted_log10"
)

save_matrix_fig(
    B_top1,
    title="Top 1% strongest connections (binary)",
    cmap="gray_r",
    vmin=0,
    vmax=1,
    with_colorbar=False,
    out_stem=out_dir / "MSA_G_top1_binary"
)


# (A) Global top 5%
save_matrix_fig(
    Wlog_top5,
    title="Top 5% strongest connections (weighted)",
    cmap=orange_white,
    vmin=vmin,
    vmax=vmax,
    with_colorbar=True,
    out_stem=out_dir / "MSA_A_top5_weighted_log10"
)

save_matrix_fig(
    B_top5,
    title="Top 5% strongest connections (binary)",
    cmap="gray_r",
    vmin=0,
    vmax=1,
    with_colorbar=False,
    out_stem=out_dir / "MSA_A_top5_binary"
)


# (B) Global top 5% minus global top 1%
save_matrix_fig(
    Wlog_top5_minus,
    title="Top 5% minus top 1% strongest connections (weighted)",
    cmap=orange_white,
    vmin=vmin,
    vmax=vmax,
    with_colorbar=True,
    out_stem=out_dir / "MSA_B_top5_minus_top1_weighted_log10"
)

save_matrix_fig(
    B_top5_minus,
    title="Top 5% minus top 1% strongest connections (binary)",
    cmap="gray_r",
    vmin=0,
    vmax=1,
    with_colorbar=False,
    out_stem=out_dir / "MSA_B_top5_minus_top1_binary"
)

print("Done. Saved to:", out_dir)


# =========================
# Save global Top 5% minus global Top 1% matrix to CSV
# =========================
W_top5_minus = np.zeros_like(W, dtype=float)

mask5_minus = mask5_minus_flat.reshape(W.shape)
W_top5_minus[mask5_minus] = W[mask5_minus]

csv_out_path = (
    data_dir
    / "MSA_Top5_minus_Top1_Connection_Data.csv"
)

np.savetxt(
    csv_out_path,
    W_top5_minus,
    delimiter=","
)

print("Saved Top 5% minus Top 1% matrix to:")
print(csv_out_path)

Done. Saved to: C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_Key_Connectomes\Matrix_visualization
Saved Top 5% minus Top 1% matrix to:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Data\MSA_Top5_minus_Top1_Connection_Data.csv


In [30]:
# plot MSA key connectome fit results
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


def concordance_correlation_coefficient(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    n = len(x)
    if n < 2:
        return np.nan

    mx, my = x.mean(), y.mean()
    vx, vy = x.var(ddof=1), y.var(ddof=1)

    if vx == 0 and vy == 0:
        return np.nan

    cov = np.cov(x, y, ddof=1)[0, 1]

    ccc = (
        2 * cov
        / (vx + vy + (mx - my) ** 2)
    )

    return ccc


def compute_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    r, p = stats.pearsonr(x, y)
    ccc = concordance_correlation_coefficient(x, y)
    df = n - 2

    return r, p, ccc, df


# =========================
# Inputs
# =========================
# Jupyter Notebook working directory:
# Code/Figures Python
project_dir = Path.cwd()

data_dir = project_dir / "Data"
results_dir = project_dir / "Results"

xlsx_path = data_dir / "MSA_Key_connectomes.xlsx"

SHEET_NAME = "MSA_Top5minusTop1_Res_Plot"

OUT_DIR = (
    results_dir
    / "MSA_Key_Connectomes"
    / "key_connectome_fit_top5minustop1"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Same axis-square helper
# =========================
def _apply_axis_square_same_scale(ax, x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if x.size == 0 or y.size == 0:
        return

    mn = min(x.min(), y.min())
    mx = max(x.max(), y.max())

    pad = 0.03 * (mx - mn) if mx > mn else 0.5

    mn -= pad
    mx += pad

    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)
    ax.set_aspect("equal", adjustable="box")


def _force_ticks_out(ax, length=4, width=1.0):
    """Force visible ticks pointing outward."""

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=length,
        width=width
    )

    ax.tick_params(
        axis="both",
        which="minor",
        direction="out",
        length=max(2, length - 2),
        width=width
    )


# =========================
# Read + strict log10
# =========================
def read_xy_and_log10(file_path, sheet_name):
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=0
    )

    if (
        ("Real3" in df.columns)
        and
        ("Predict3" in df.columns)
    ):
        x_raw = pd.to_numeric(
            df["Real3"],
            errors="coerce"
        ).to_numpy(dtype=float)

        y_raw = pd.to_numeric(
            df["Predict3"],
            errors="coerce"
        ).to_numpy(dtype=float)

    else:
        x_raw = pd.to_numeric(
            df.iloc[:, 2],
            errors="coerce"
        ).to_numpy(dtype=float)

        y_raw = pd.to_numeric(
            df.iloc[:, 3],
            errors="coerce"
        ).to_numpy(dtype=float)

    # Strict log10 transformation
    x = np.log10(x_raw)
    y = np.log10(y_raw)

    m = np.isfinite(x) & np.isfinite(y)

    x = x[m]
    y = y[m]

    df_xy = pd.DataFrame({
        "Real3": x,
        "Predict3": y
    })

    return df_xy


# =========================
# Draw: jointplot + marginals + axis square
# =========================
def draw_joint_with_marginals(df_xy, name_for_save):

    sns.set_style(
        "white",
        rc={
            "xtick.bottom": True,
            "ytick.left": True,
            "xtick.major.size": 4,
            "ytick.major.size": 4,
            "xtick.major.width": 1.0,
            "ytick.major.width": 1.0,
            "xtick.direction": "out",
            "ytick.direction": "out",
        }
    )

    sns.set_palette(
        sns.color_palette("deep")
    )

    fig = sns.jointplot(
        data=df_xy,
        x="Real3",
        y="Predict3",
        kind="reg"
    )

    fig.ax_joint.set_xlabel(
        "Log(Pathologies)",
        fontweight="bold",
        fontsize=14
    )

    fig.ax_joint.set_ylabel(
        "Log(Predicted)",
        fontweight="bold",
        fontsize=14
    )

    _apply_axis_square_same_scale(
        fig.ax_joint,
        df_xy["Real3"],
        df_xy["Predict3"]
    )

    fig.plot_marginals(
        sns.histplot,
        kde=True
    )

    # Statistics calculated using the same log10-transformed data
    r, p, ccc, df = compute_stats(
        df_xy["Real3"],
        df_xy["Predict3"]
    )

    stat_txt = (
        f"R = {r:.3f}\n"
        f"p = {p:.2e}\n"
        f"df = {df:d}\n"
        f"CCC = {ccc:.3f}"
    )

    fig.ax_joint.text(
        0.04,
        0.96,
        stat_txt,
        transform=fig.ax_joint.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="none",
            alpha=0.85
        )
    )

    fig.ax_joint.tick_params(
        axis="both",
        which="both",
        direction="out",
        bottom=True,
        left=True,
        length=4,
        width=1.0
    )

    fig.ax_joint.xaxis.set_ticks_position("bottom")
    fig.ax_joint.yaxis.set_ticks_position("left")

    save_pdf = (
        OUT_DIR
        / f"{name_for_save}_joint_with_marginals.pdf"
    )

    save_tif = (
        OUT_DIR
        / f"{name_for_save}_joint_with_marginals.tif"
    )

    fig.savefig(
        save_pdf,
        format="pdf"
    )

    fig.savefig(
        save_tif,
        format="tif",
        dpi=300,
        facecolor="white",
        transparent=False
    )

    plt.close(fig.fig)

    print("Saved:")
    print(save_pdf)
    print(save_tif)


# =========================
# Run
# =========================
df_xy = read_xy_and_log10(
    xlsx_path,
    SHEET_NAME
)

draw_joint_with_marginals(
    df_xy,
    name_for_save="MSA_Top5minusTop1_Res_Plot"
)

C:\Users\forge\AppData\Local\Temp\ipykernel_5712\468810916.py:163: RuntimeWarning: divide by zero encountered in log10
  x = np.log10(x_raw)
C:\Users\forge\AppData\Local\Temp\ipykernel_5712\468810916.py:164: RuntimeWarning: divide by zero encountered in log10
  y = np.log10(y_raw)


Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_Key_Connectomes\key_connectome_fit_top5minustop1\MSA_Top5minusTop1_Res_Plot_joint_with_marginals.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_Key_Connectomes\key_connectome_fit_top5minustop1\MSA_Top5minusTop1_Res_Plot_joint_with_marginals.tif
